In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully!')

Mounted at /content/drive
Google Drive mounted successfully!


In [ ]:
!pip install -q tiktoken datasets huggingface_hub

In [ ]:
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM     : {vram:.1f} GB")
    # Enable TF32 for faster matmul on Ampere GPUs (no quality loss)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print("✓ Dependencies ready")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : NVIDIA A100-SXM4-40GB
VRAM     : 42.4 GB
✓ Dependencies ready


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

REPO_URL = "https://github.com/Adithya-Rama/nano-llm.git"
LOCAL_DIR = "/content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code"

if not os.path.exists(LOCAL_DIR):
    !git clone "{REPO_URL}" "{LOCAL_DIR}"
else:
    !cd "{LOCAL_DIR}" && git pull

os.chdir(f"{LOCAL_DIR}")
print(f"✓ Working directory: {os.getcwd()}")

# DRIVE_CKPT = "/content/drive/My Drive/COMP8650/Assgn-1/nano-llm/nanogpt_rocstories_checkpoints"
DRIVE_CKPT = "/content/drive/My Drive/COMP8650/Assgn-1/nano-llm/code/out-rocstories-combined"
os.makedirs(DRIVE_CKPT, exist_ok=True)

print(f"✓ Checkpoint dir (Drive): {DRIVE_CKPT}")

Mounted at /content/drive
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
chdir: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 5.68 KiB | 1024 bytes/s, done.
From https://github.com/Adithya-Rama/nano-llm
   d36472c..982c409  main       -> origin/main
Updating d36472c..982c409
Fast-forward
 colab_setup.py              | 447 +++++++++++++++++++++++++++++++++++---------
 data/tinystories/prepare.py |   2 +-
 2 files changed, 358 insertions(+), 91 deletions(-)
✓ Working directory: /content/drive/.shortcut-targets-by-id/1f5p9LYFtBc18-IpN69Wl8SYpii0xYx1V/COMP8650/Assgn-1/nano-llm/code
✓ Checkpoint dir (Drive): /content/

In [ ]:
import os

# ── Config: match whichever config you want to train ────────────────────────
CONFIG_FILE  = "config/train_rocstories_combined.py"  # ← change if needed
OUT_DIR      = "out-rocstories-combined"              # ← must match config

os.chdir(LOCAL_DIR)

# ── Locations to check for an existing checkpoint ───────────────────────────
local_ckpt = os.path.join(OUT_DIR, "ckpt.pt")
drive_ckpt = os.path.join(DRIVE_CKPT, "ckpt.pt")

if os.path.exists(local_ckpt):
    INIT_FROM = "resume"
    print(f"✓ Checkpoint found locally:  {local_ckpt}")
elif os.path.exists(drive_ckpt):
    # Restore from Drive into the expected out-dir so train.py can find it
    os.makedirs(OUT_DIR, exist_ok=True)
    import shutil
    shutil.copy(drive_ckpt, local_ckpt)
    INIT_FROM = "resume"
    print(f"✓ Checkpoint restored from Drive → {local_ckpt}")
    # Also copy train_log.jsonl if present (for learning-curve continuity)
    drive_log = os.path.join(DRIVE_CKPT, "train_log.jsonl")
    if os.path.exists(drive_log):
        shutil.copy(drive_log, os.path.join(OUT_DIR, "train_log.jsonl"))
        print("  (train_log.jsonl restored too)")
else:
    INIT_FROM = "scratch"
    print("ℹ  No checkpoint found — will train from scratch.")

print(f"\n→ INIT_FROM = '{INIT_FROM}'")
print(f"→ CONFIG    = '{CONFIG_FILE}'")
print("\nCopy the variables above into CELL 5, or just run CELL 5 next —")
print("it reads INIT_FROM automatically if you keep both cells in the same session.")

✓ Checkpoint found locally:  out-rocstories-combined/ckpt.pt

→ INIT_FROM = 'resume'
→ CONFIG    = 'config/train_rocstories_combined.py'

Copy the variables above into CELL 5, or just run CELL 5 next —
it reads INIT_FROM automatically if you keep both cells in the same session.


In [ ]:
import os
os.chdir(LOCAL_DIR)

# ── Choose config (keep in sync with CELL 5a) ───────────────────────────────
CONFIG_FILE = "config/train_rocstories_combined.py"  # ← change if needed
OUT_DIR     = "out-rocstories-combined"              # ← must match config

# ── Auto-detect whether to resume or start fresh ────────────────────────────
try:
    # INIT_FROM may already be set by CELL 5a in this session
    print(f"Using INIT_FROM='{INIT_FROM}' (set by CELL 5a)")
except NameError:
    # CELL 5a wasn't run (e.g. fresh session) — check ourselves
    ckpt_path = os.path.join(OUT_DIR, "ckpt.pt")
    INIT_FROM = "resume" if os.path.exists(ckpt_path) else "scratch"
    if INIT_FROM == "resume":
        print(f"✓ ckpt.pt found in {OUT_DIR} — resuming from checkpoint")
    else:
        print("ℹ  No checkpoint found — starting from scratch")

print(f"\n→ Running: python train.py {CONFIG_FILE} --init_from={INIT_FROM}\n")

!python train.py {CONFIG_FILE} --init_from={INIT_FROM}

# ── Backup checkpoint to Drive after training finishes ───────────────────────
# DRIVE_CKPT = "/content/drive/MyDrive/nanogpt_rocstories_checkpoints"
if os.path.exists(os.path.join(OUT_DIR, "ckpt.pt")):
    import shutil
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    try:
        shutil.copy(os.path.join(OUT_DIR, "ckpt.pt"), os.path.join(DRIVE_CKPT, "ckpt.pt"))
        log_src = os.path.join(OUT_DIR, "train_log.jsonl")
        if os.path.exists(log_src):
            shutil.copy(log_src, os.path.join(DRIVE_CKPT, "train_log.jsonl"))
        print(f"✓ Checkpoint backed up to Drive: {DRIVE_CKPT}")
    except shutil.SameFileError:
        print("✓ Checkpoint already on Drive (repo is on Drive) — no backup needed")

print("✓ Training complete (or interrupted — re-run to resume)")

Using INIT_FROM='resume' (set by CELL 5a)

→ Running: python train.py config/train_rocstories_combined.py --init_from=resume

Overriding config with config/train_rocstories_combined.py:
# ============================================================
# config/train_rocstories_combined.py
#
# Train on combined ROCStories + TinyStories dataset.
# This provides ~60-80M tokens of narrative text instead of
# just ~2.25M from ROCStories alone, significantly reducing
# overfitting and improving generalization.
#
# Use this config for Task 3 (best checkpoint submission).
# The model will learn general story patterns from TinyStories
# and specific 5-sentence narrative arcs from ROCStories.
#
# Data preparation:
#   python data/rocstories/prepare.py
#   python data/tinystories/prepare.py
#   python data/combined/prepare.py
#
# Usage:
#   python train.py config/train_rocstories_combined.py
# ============================================================

# ── I/O ────────────────────────────────────

In [ ]:
import torch, sys
sys.path.insert(0, LOCAL_DIR)

from model import GPT, GPTConfig

# Build the ~152M parameter model and estimate VRAM
cfg = GPTConfig(
    n_layer=12, n_head=12, n_embd=768, block_size=256, vocab_size=50304,
    bias=False, dropout=0.1,
    use_rmsnorm=True, use_rope=True, use_swiglu=True, use_qk_norm=True
)
model = GPT(cfg)
n_params = sum(p.numel() for p in model.parameters()) / 1e6

# Rough VRAM estimate (bfloat16 weights + fp32 Adam states + activations)
batch_size, seq_len = 16, 256
bytes_model   = n_params * 1e6 * 2          # bfloat16
bytes_adam    = n_params * 1e6 * 8          # 2× fp32 Adam states
bytes_act     = batch_size * seq_len * cfg.n_embd * cfg.n_layer * 2  # rough
total_gb = (bytes_model + bytes_adam + bytes_act) / 1e9

print(f"Model parameters : {n_params:.1f}M")
print(f"Estimated VRAM   : {total_gb:.1f} GB  (rough upper bound)")

if torch.cuda.is_available():
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    avail_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f"Available VRAM   : {free_gb:.1f} GB / {avail_gb:.1f} GB total")
    if total_gb < free_gb * 0.85:
        print("✓ Model fits with headroom")
    else:
        print("⚠  Tight fit! Consider reducing batch_size to 8.")

del model
torch.cuda.empty_cache()

number of parameters: 151.90M
Model parameters : 151.9M
Estimated VRAM   : 1.6 GB  (rough upper bound)
Available VRAM   : 42.0 GB / 42.4 GB total
✓ Model fits with headroom


In [ ]:
from google.colab import userdata
import os

# Get the HF_TOKEN from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Set it as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN successfully set as an environment variable.")

HF_TOKEN successfully set as an environment variable.


In [ ]:
# Install huggingface_hub if not already installed
# !pip install -q huggingface_hub

from huggingface_hub import login

# Login using the token from the environment variable
login(token=os.environ['HF_TOKEN'])

print("Successfully logged in to Hugging Face.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully logged in to Hugging Face.


In [ ]:
import os, sys
os.chdir(LOCAL_DIR)

import numpy as np

# 1. Prepare ROCStories — plain text (required for Task 1 + Task 3)
print("=" * 60)
print("Preparing ROCStories (plain format)...")
print("=" * 60)
!python data/rocstories/prepare.py

# 2. Prepare ROCStories — structured format (Task 2 experiment)
print()
print("=" * 60)
print("Preparing ROCStories (structured format with sentence tags)...")
print("=" * 60)
!python data/rocstories/prepare.py --structured

# 3. Prepare TinyStories (Task 2 experiment — extra data)
print()
print("=" * 60)
print("Preparing TinyStories (this may take a few minutes)...")
print("=" * 60)
!python data/tinystories/prepare.py

# 4. Create combined dataset (Task 2 experiment)
print()
print("=" * 60)
print("Creating combined dataset...")
print("=" * 60)
!python data/combined/prepare.py

# Verify all outputs
print()
print("=" * 60)
print("Dataset Summary")
print("=" * 60)
for ds_name in ['rocstories', 'rocstories_structured', 'tinystories', 'combined']:
    for split in ['train', 'val']:
        fpath = f'data/{ds_name}/{split}.bin'
        if os.path.exists(fpath):
            arr = np.fromfile(fpath, dtype=np.uint16)
            print(f"  {ds_name}/{split}.bin: {len(arr):,} tokens")

print("\\n✓ All datasets ready")

Preparing ROCStories (plain format)...
[prepare] Trying to load: mintujupally/ROCStories ...
README.md: 100% 256/256 [00:00<00:00, 1.12MB/s]
Repo card metadata block was not found. Setting CardData to empty.
train.txt: 100% 18.1M/18.1M [00:01<00:00, 11.6MB/s]
test.txt: 4.52MB [00:00, 16.6MB/s]
Generating train split: 100% 78528/78528 [00:00<00:00, 249721.03 examples/s]
Generating test split: 100% 19633/19633 [00:00<00:00, 371709.98 examples/s]
[prepare] Columns: ['text']
[prepare] Loaded 78,528 stories from mintujupally/ROCStories (plain format)
[prepare] Split: 70,676 train | 7,852 val

[prepare] Sample story:
The boy went to a video arcade. He played his favorite machine. His games didn't go very well. He told the owner about his experience. The owner explained that he had made the game settings harder.

[prepare] train.bin: 3,701,102 tokens  (avg 52.4 tok/story)  -> /content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code/data/rocstories/train.bin
[prepare] val.bin: 410,039 tokens  (av

In [ ]:
import os
os.chdir(LOCAL_DIR)

# ── OPTION A: Train on ROCStories only (Task 1) ─────────────────────────────
# !python train.py config/train_rocstories.py

# ── OPTION B: Train on combined ROCStories+TinyStories (best for Task 3) ────
!python train.py config/train_rocstories_combined.py

# ── To resume after crash: ───────────────────────────────────────────────────
# !python train.py config/train_rocstories_combined.py --init_from=resume

print("✓ Training complete (or interrupted — re-run to resume)")

Overriding config with config/train_rocstories_combined.py:
# ============================================================
# config/train_rocstories_combined.py
#
# Train on combined ROCStories + TinyStories dataset.
# This provides ~60-80M tokens of narrative text instead of
# just ~2.25M from ROCStories alone, significantly reducing
# overfitting and improving generalization.
#
# Use this config for Task 3 (best checkpoint submission).
# The model will learn general story patterns from TinyStories
# and specific 5-sentence narrative arcs from ROCStories.
#
# Data preparation:
#   python data/rocstories/prepare.py
#   python data/tinystories/prepare.py
#   python data/combined/prepare.py
#
# Usage:
#   python train.py config/train_rocstories_combined.py
# ============================================================

# ── I/O ──────────────────────────────────────────────────────────────────────
out_dir    = 'out-rocstories-combined'
eval_interval  = 500
log_interval   = 10
eval_iters 

In [ ]:
import os

# Read the config to see exactly what max_iters is set to
config_path = os.path.join(LOCAL_DIR, 'config/train_rocstories_combined.py')

with open(config_path, 'r') as f:
    config_content = f.read()

# Extract max_iters and gradient_accumulation_steps
import re
max_iters = re.findall(r'max_iters\s*=\s*(\d+)', config_content)
grad_accum = re.findall(r'gradient_accumulation_steps\s*=\s*(\d+)', config_content)

print(f"Configured max_iters: {max_iters[0] if max_iters else 'Not found'}")
print(f"Gradient Accumulation Steps: {grad_accum[0] if grad_accum else 'Not found'}")

# Calculate total 'Steps' (weight updates)
if max_iters and grad_accum:
    total_steps = int(max_iters[0])
    print(f"Total weight update steps planned: {total_steps}")

Configured max_iters: 20000
Gradient Accumulation Steps: 8
Total weight update steps planned: 20000


In [ ]:
import os
# Use the LOCAL_DIR variable defined earlier to ensure we are in the correct directory
os.chdir(LOCAL_DIR)

# Change out_dir to match whichever config you trained with
OUT_DIR = "out-rocstories-combined"  # or "out-rocstories"

print("=" * 60)
print("PERPLEXITY EVALUATION on eval_stories.txt")
print("=" * 60)
!python eval.py --init_from=resume --out_dir={OUT_DIR} --input_file=data/rocstories/eval_stories.txt

print()
print("=" * 60)
print("STORY GENERATION from eval_prompts.txt")
print("=" * 60)
!python sample_batch.py --init_from=resume --out_dir={OUT_DIR} \
    --start="FILE:data/rocstories/eval_prompts.txt" \
    --batch_prompts=True \
    --max_new_tokens=200 \
    --output_file={OUT_DIR}/generated_stories.jsonl

PERPLEXITY EVALUATION on eval_stories.txt
Overriding: init_from = resume
Overriding: out_dir = out-rocstories-combined
Overriding: input_file = data/rocstories/eval_stories.txt
number of parameters: 151.90M
No meta.pkl found, assuming GPT-2 encodings...
Loaded 9 paragraphs from data/rocstories/eval_stories.txt (format=txt)
[preview 0] Emily forgot her umbrella before leaving for work. Dark clouds gathered as she walked to the bus stop. Halfway there, it...
[preview 1] Tom decided to cook dinner for his friends. He followed a recipe he found online. Halfway through, he realized he forgot...
[preview 2] Lily wanted to start jogging every morning. On the first day, she woke up before sunrise. The air was cold and the stree...
----- Evaluation Results -----
model           : resume
paragraphs_used : 9
paragraphs_skip : 0
pred_tokens     : 413
avg_loss        : 3.007
ppl             : 20.23

STORY GENERATION from eval_prompts.txt
Overriding: init_from = resume
Overriding: out_dir = out-rocs

In [ ]:
# Verify the files exist in the directory
print(f"Checking contents of: {os.path.join(LOCAL_DIR, OUT_DIR)}")
!ls -lh "{os.path.join(LOCAL_DIR, OUT_DIR)}"

Checking contents of: /content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code/out-rocstories-combined
total 3.6G
-rw------- 1 root root 148M Mar  6 13:45 ckpt_emergency.pt.tmp
-rw------- 1 root root 1.7G Mar  6 15:53 ckpt_final.pt
-rw------- 1 root root 1.7G Mar  6 15:54 ckpt.pt
-rw------- 1 root root 2.9K Mar  6 15:58 generated_stories.jsonl
-rw------- 1 root root 294K Mar  6 15:53 train_log.jsonl


**Tip:** If you absolutely need to see them in Drive immediately, you can force a sync by unmounting (though this will stop your ability to write until you remount):
```python
from google.colab import drive
drive.flush_and_unmount()
```
Otherwise, just waiting a few minutes usually resolves it.

In [ ]:
drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os

# 1. Remount the drive
drive.mount('/content/drive')

# 2. Check the directory again
target_dir = os.path.join(LOCAL_DIR, OUT_DIR)
if os.path.exists(target_dir):
    print(f"\n✓ SUCCESS: The folder exists at: {target_dir}")
    print("Files found:")
    !ls -lh "{target_dir}"
else:
    print(f"\n⚠ Folder not found at: {target_dir}")

## Ablation Studies - TASK 2

In [ ]:
import os, subprocess, json, time
os.chdir(LOCAL_DIR)

In [ ]:
experiments = [
    # (config_file, out_dir, description)
    ("config/train_rocstories_baseline.py",       "out-rocstories-baseline",    "A. Vanilla GPT"),
    ("config/train_rocstories_rope_only.py",      "out-rocstories-rope",        "B. +RoPE only"),
    ("config/train_rocstories_rmsnorm_swiglu.py",  "out-rocstories-ffn",        "C. +RMSNorm+SwiGLU"),
    ("config/train_rocstories_qknorm.py",          "out-rocstories-qknorm",     "D. +QK-Norm only"),
    ("config/train_rocstories.py",                 "out-rocstories",            "E. All modern (full)"),
    ("config/train_rocstories_structured.py",      "out-rocstories-structured", "F. Structured format"),
]

In [ ]:

# ── Load any previously saved results (resume support) ───────────────────────
RESULTS_FILE = "ablation_results.json"
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        results = json.load(f)
    print(f"✓ Loaded {len(results)} previous results from {RESULTS_FILE}")
else:
    results = {}

In [ ]:
total = len(experiments)
overall_start = time.time()
completed = 0
skipped = 0

for i, (config, out_dir, desc) in enumerate(experiments, 1):
    exp_start = time.time()
    ckpt_path = os.path.join(out_dir, "ckpt.pt")
    stories_path = os.path.join(out_dir, "generated_stories.jsonl")

    print(f"\\n{'━'*70}")
    print(f"  EXPERIMENT {i}/{total}: {desc}")
    print(f"  Config : {config}")
    print(f"  Out dir: {out_dir}")
    print(f"  Time   : {time.strftime('%H:%M:%S')}")
    print(f"{'━'*70}")

    # ── Phase 1: TRAIN ───────────────────────────────────────────────────────
    if os.path.exists(ckpt_path):
        print(f"  ⏩ SKIP training — checkpoint already exists: {ckpt_path}")
        skipped += 1
    else:
        print(f"  🚀 [1/3] Training {desc}...")
        train_result = subprocess.run(
            ["python", "train.py", config],
            check=False
        )
        if train_result.returncode != 0:
            print(f"  ❌ Training FAILED for {desc} (exit code {train_result.returncode})")
            print(f"     Skipping evaluation — continuing to next experiment.")
            continue
        elapsed = time.time() - exp_start
        print(f"  ✓ Training done ({elapsed/60:.1f} min)")

    # ── Phase 2: EVALUATE PERPLEXITY ─────────────────────────────────────────
    print(f"  📊 [2/3] Evaluating perplexity...")
    eval_result = subprocess.run(
        ["python", "eval.py",
         "--init_from=resume", f"--out_dir={out_dir}",
         "--input_file=data/rocstories/eval_stories.txt"],
        capture_output=True, text=True
    )
    # Print eval output
    if eval_result.stdout:
        for line in eval_result.stdout.strip().split('\\n'):
            print(f"     {line}")
    if eval_result.stderr:
        # Only print non-empty stderr lines that aren't just warnings
        for line in eval_result.stderr.strip().split('\\n'):
            if line.strip() and 'warning' not in line.lower():
                print(f"     [stderr] {line}")
    # Extract perplexity
    ppl_found = False
    for line in eval_result.stdout.split('\\n'):
        if 'ppl' in line.lower() or 'perplexity' in line.lower():
            results[desc] = line.strip()
            ppl_found = True
    if not ppl_found:
        results[desc] = f"(eval completed, check {out_dir} manually)"
    print(f"  ✓ Eval done")

    # ── Phase 3: GENERATE STORIES ────────────────────────────────────────────
    print(f"  ✍️  [3/3] Generating stories for quality evaluation...")
    gen_result = subprocess.run([
        "python", "sample_batch.py",
        "--init_from=resume", f"--out_dir={out_dir}",
        "--start=FILE:data/rocstories/eval_prompts.txt",
        "--batch_prompts=True", "--max_new_tokens=200",
        f"--output_file={out_dir}/generated_stories.jsonl"
    ], check=False)
    if gen_result.returncode == 0:
        print(f"  ✓ Stories saved to {out_dir}/generated_stories.jsonl")
    else:
        print(f"  ⚠ Story generation had issues (exit code {gen_result.returncode})")

    # ── Save results after each experiment (crash resilience) ────────────────
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2)

    completed += 1
    elapsed = time.time() - exp_start
    total_elapsed = time.time() - overall_start
    remaining = total - i
    avg_per_exp = total_elapsed / (completed + skipped) if (completed + skipped) > 0 else 0
    eta_min = (remaining * avg_per_exp) / 60

    print(f"\\n  ⏱  This experiment: {elapsed/60:.1f} min")
    print(f"  📈 Progress: {i}/{total} done | {remaining} remaining | ETA: ~{eta_min:.0f} min")

\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXPERIMENT 1/6: A. Vanilla GPT
  Config : config/train_rocstories_baseline.py
  Out dir: out-rocstories-baseline
  Time   : 05:31:55
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🚀 [1/3] Training A. Vanilla GPT...
  ✓ Training done (93.1 min)
  📊 [2/3] Evaluating perplexity...
     Overriding: init_from = resume
Overriding: out_dir = out-rocstories-baseline
Overriding: input_file = data/rocstories/eval_stories.txt
number of parameters: 123.59M
No meta.pkl found, assuming GPT-2 encodings...
Loaded 9 paragraphs from data/rocstories/eval_stories.txt (format=txt)
[preview 0] Emily forgot her umbrella before leaving for work. Dark clouds gathered as she walked to the bus stop. Halfway there, it...
[preview 1] Tom decided to cook dinner for his friends. He followed a recipe he found online. Halfway through, he realized he forgot...
[preview 2] Lily wanted to start jogging every morning. On t

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
total_time = time.time() - overall_start
print(f"\\n{'━'*70}")
print(f"  ABLATION COMPLETE — {completed} trained, {skipped} skipped (had checkpoint)")
print(f"  Total time: {total_time/60:.1f} min ({total_time/3600:.1f} hrs)")
print(f"{'━'*70}")
print()
print(f"{'='*60}")
print(f"  ABLATION RESULTS SUMMARY")
print(f"{'='*60}")
for desc, val in results.items():
    print(f"  {desc:40s}: {val}")
print(f"{'='*60}")
print(f"\\n✓ Results saved to {RESULTS_FILE}")
print(f"  Run CELL 7b next for graphs and analysis!")

\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ABLATION COMPLETE — 6 trained, 0 skipped (had checkpoint)
  Total time: 623.2 min (10.4 hrs)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ABLATION RESULTS SUMMARY
  A. Vanilla GPT                          : Overriding: init_from = resume
Overriding: out_dir = out-rocstories-baseline
Overriding: input_file = data/rocstories/eval_stories.txt
number of parameters: 123.59M
No meta.pkl found, assuming GPT-2 encodings...
Loaded 9 paragraphs from data/rocstories/eval_stories.txt (format=txt)
[preview 0] Emily forgot her umbrella before leaving for work. Dark clouds gathered as she walked to the bus stop. Halfway there, it...
[preview 1] Tom decided to cook dinner for his friends. He followed a recipe he found online. Halfway through, he realized he forgot...
[preview 2] Lily wanted to start jogging every morning. On the first day, she woke up before sunrise. The air was cold and the stree..

### Plot learning curves + Story quality metrics + Bar charts

In [ ]:
import os, json
os.chdir(LOCAL_DIR)

!pip install -q matplotlib

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
# ── Discover which experiments actually completed ────────────────────────────
all_experiments = [
    ("out-rocstories-baseline",    "Baseline",        "out-rocstories-baseline/train_log.jsonl"),
    ("out-rocstories-rope",        "+RoPE",           "out-rocstories-rope/train_log.jsonl"),
    ("out-rocstories-ffn",         "+RMSNorm+SwiGLU", "out-rocstories-ffn/train_log.jsonl"),
    ("out-rocstories-qknorm",      "+QK-Norm",        "out-rocstories-qknorm/train_log.jsonl"),
    ("out-rocstories",             "All Modern",      "out-rocstories/train_log.jsonl"),
    ("out-rocstories-structured",  "Structured Fmt",  "out-rocstories-structured/train_log.jsonl"),
]

available = [(d, l, p) for d, l, p in all_experiments if os.path.exists(p)]
print(f"Found {len(available)}/{len(all_experiments)} completed experiments:\\n")
for out_dir, label, log_path in available:
    size_mb = os.path.getsize(log_path) / 1e6
    print(f"  ✓ {label:20s} — {log_path} ({size_mb:.1f} MB)")
missing = [(d, l, p) for d, l, p in all_experiments if not os.path.exists(p)]
for out_dir, label, log_path in missing:
    print(f"  ✗ {label:20s} — NOT FOUND (skipping)")

Found 6/6 completed experiments:\n
  ✓ Baseline             — out-rocstories-baseline/train_log.jsonl (0.3 MB)
  ✓ +RoPE                — out-rocstories-rope/train_log.jsonl (0.3 MB)
  ✓ +RMSNorm+SwiGLU      — out-rocstories-ffn/train_log.jsonl (0.3 MB)
  ✓ +QK-Norm             — out-rocstories-qknorm/train_log.jsonl (0.3 MB)
  ✓ All Modern           — out-rocstories/train_log.jsonl (0.3 MB)
  ✓ Structured Fmt       — out-rocstories-structured/train_log.jsonl (0.3 MB)


In [ ]:
if len(available) == 0:
    print("\\n❌ No completed experiments found! Run previous cell first.")
else:
    # ── 1. Architecture ablation (exclude structured format) ─────────────
    arch_exps = [(d, l, p) for d, l, p in available if "structured" not in d]
    if len(arch_exps) >= 2:
        print(f"\\n{'='*60}")
        print("1. Architecture ablation learning curves...")
        print(f"{'='*60}")
        log_args = []
        labels = []
        for _, label, log_path in arch_exps:
            log_args.extend(["--log", log_path])
            labels.append(label)
        !python plot_training.py {' '.join(log_args)} \\
            --labels "{','.join(labels)}" \\
            --output ablation_curves.png \\
            --title "Task 2: Architecture Ablation — Validation Loss"
    else:
        print("\\n⚠ Not enough architecture experiments for ablation plot")

    # ── 2. Full 6-way comparison (if structured exists too) ──────────────
    if len(available) > len(arch_exps):
        print(f"\\n{'='*60}")
        print("2. Full 6-way comparison (including structured format)...")
        print(f"{'='*60}")
        log_args = []
        labels = []
        for _, label, log_path in available:
            log_args.extend(["--log", log_path])
            labels.append(label)
        !python plot_training.py {' '.join(log_args)} \\
            --labels "{','.join(labels)}" \\
            --output ablation_curves_full.png \\
            --title "Task 2: All Experiments — Validation Loss"

    # ── 3. Single best model training curve ──────────────────────────────
    best_log = "out-rocstories/train_log.jsonl"
    if not os.path.exists(best_log):
        best_log = "out-rocstories-combined/train_log.jsonl"
    if os.path.exists(best_log):
        print(f"\\n{'='*60}")
        print(f"3. Best model training curve ({best_log})...")
        print(f"{'='*60}")
        !python plot_training.py \\
            --log {best_log} \\
            --output training_curve.png \\
            --title "Task 1: Training Curves (152M All Modern)"

    # ── 4. Bar chart: final val loss per experiment ──────────────────────
    print(f"\\n{'='*60}")
    print("4. Final validation loss bar chart...")
    print(f"{'='*60}")
    bar_labels = []
    bar_vals = []
    for out_dir, label, log_path in available:
        with open(log_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        # Find last line with val_loss
        for line in reversed(lines):
            entry = json.loads(line)
            if 'val_loss' in entry and entry['val_loss'] is not None:
                bar_labels.append(label)
                bar_vals.append(entry['val_loss'])
                break
    if bar_vals:
        fig, ax = plt.subplots(figsize=(9, 5))
        colors = ['#E53935', '#1E88E5', '#43A047', '#FB8C00', '#8E24AA', '#00ACC1']
        bars = ax.bar(bar_labels, bar_vals, color=colors[:len(bar_vals)], edgecolor='white', linewidth=1.5)
        ax.set_ylabel('Final Validation Loss', fontsize=12)
        ax.set_title('Task 2: Final Validation Loss by Configuration', fontsize=14, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        # Add value labels on bars
        for bar, val in zip(bars, bar_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        plt.xticks(rotation=15, ha='right', fontsize=10)
        plt.tight_layout()
        plt.savefig('ablation_bar_chart.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✓ Saved: ablation_bar_chart.png")

        # Print table too
        print(f"\\n  {'Config':25s} {'Val Loss':>10s}")
        print(f"  {'-'*37}")
        for lbl, val in sorted(zip(bar_labels, bar_vals), key=lambda x: x[1]):
            print(f"  {lbl:25s} {val:10.4f}")

    # ── 5. Story quality comparison ──────────────────────────────────────
    quality_inputs = []
    quality_labels = []
    for out_dir, label, _ in available:
        stories_path = os.path.join(out_dir, "generated_stories.jsonl")
        if os.path.exists(stories_path):
            quality_inputs.extend(["--input", stories_path])
            quality_labels.append(label)
        else:
            print(f"  ⚠ No generated stories for {label} — skipping quality eval")

    if quality_labels:
        print(f"\\n{'='*60}")
        print(f"5. Story quality metrics ({len(quality_labels)} models)...")
        print(f"{'='*60}")
        !python eval_story_quality.py {' '.join(quality_inputs)} \\
            --labels "{','.join(quality_labels)}"

    # ── Summary ──────────────────────────────────────────────────────────
    print(f"\\n{'━'*60}")
    print("  OUTPUTS GENERATED:")
    for fname in ['ablation_curves.png', 'ablation_curves_full.png',
                   'training_curve.png', 'ablation_bar_chart.png']:
        if os.path.exists(fname):
            print(f"    ✓ {fname}")
        else:
            print(f"    ✗ {fname} (not generated)")
    print(f"{'━'*60}")

\n============================================================
1. Architecture ablation learning curves...
usage: plot_training.py [-h] --log LOG [--labels LABELS] [--output OUTPUT]
                        [--title TITLE]
plot_training.py: error: unrecognized arguments:      
\n============================================================
2. Full 6-way comparison (including structured format)...
usage: plot_training.py [-h] --log LOG [--labels LABELS] [--output OUTPUT]
                        [--title TITLE]
plot_training.py: error: unrecognized arguments:      
\n============================================================
3. Best model training curve (out-rocstories/train_log.jsonl)...
usage: plot_training.py [-h] --log LOG [--labels LABELS] [--output OUTPUT]
                        [--title TITLE]
plot_training.py: error: unrecognized arguments:      
\n============================================================
4. Final validation loss bar chart...
  ✓ Saved: ablation_bar_chart.png